In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np
print("libraries loaded")

In [ ]:
def residual_block(x, filters=64):
    """
    One RNB block from the paper:
    Two conv layers + shortcut connection + maxpool
    """
    # Save input for shortcut
    shortcut = x

    # First conv layer
    x = layers.Conv2D(filters, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # Second conv layer
    x = layers.Conv2D(filters, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)

    # Shortcut connection — add input back
    # If channel dimensions don't match, use 1x1 conv to align
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1, 1), padding="same")(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)

    # MaxPool + BatchNorm
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.BatchNormalization()(x)

    return x

In [ ]:
def build_m64rn4():
    """
    M64RN4: 4 residual blocks of 64 channels + FC layers
    Input: 49x49 single channel SAR patch
    Output: wind direction in radians (aliased, 180° ambiguity)
    """
    inputs = tf.keras.Input(shape=(49, 49, 1))

    # 4 residual blocks
    x = residual_block(inputs, filters=64)
    x = residual_block(x,      filters=64)
    x = residual_block(x,      filters=64)
    x = residual_block(x,      filters=64)

    # Flatten
    x = layers.Flatten()(x)

    # Fully connected layers
    x = layers.Dense(512)(x)
    x = layers.ReLU()(x)
    x = layers.BatchNormalization()(x)

    x = layers.Dense(128)(x)
    x = layers.ReLU()(x)
    x = layers.BatchNormalization()(x)

    x = layers.Dense(32)(x)
    x = layers.ReLU()(x)
    x = layers.BatchNormalization()(x)

    # Output: single angle in radians
    outputs = layers.Dense(1)(x)

    model = Model(inputs, outputs, name="M64RN4")
    return model

In [ ]:
def wind_direction_loss(y_true, y_pred):
    """
    L = 1 - cos²(θ_predicted - θ_target)
    Handles circular nature of angles.
    Returns 0 when difference is 0° or 180°.
    """
    diff = y_pred - y_true
    return 1.0 - tf.square(tf.cos(diff))

In [ ]:
import numpy as np

np.random.seed(42)
N = 1000

# Fake SAR patches — random noise, z-score normalised
X_fake = np.random.randn(N, 49, 49, 1).astype(np.float32)

# Fake direction labels in radians (-π/2 to π/2)
y_fake = np.random.uniform(-np.pi/2, np.pi/2, N).astype(np.float32)

# 90/10 split like the paper
split = int(0.9 * N)
X_train, X_val = X_fake[:split], X_fake[split:]
y_train, y_val = y_fake[:split], y_fake[split:]

print(f"Training samples:   {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Patch shape:        {X_train[0].shape}")
print(f"Label range:        {y_train.min():.3f} to {y_train.max():.3f} radians")

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True,
    verbose=1,
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1,
)

print(f"\nBest val loss: {min(history.history['val_loss']):.4f}")

In [ ]:
import numpy as np
from itertools import product as iproduct

def extract_patches(image_array, patch_size=49):
    """
    Cut a 2D SAR image into non-overlapping 49×49 patches.
    Returns array of shape (N, 49, 49, 1)
    
    image_array: 2D numpy array of shape (H, W) — VV backscatter in dB
    """
    H, W = image_array.shape
    patches = []
    coords  = []  # centre lat/lon of each patch (needed to match ERA5)

    # Step through image in 49-pixel steps
    for row in range(0, H - patch_size + 1, patch_size):
        for col in range(0, W - patch_size + 1, patch_size):
            patch = image_array[row:row+patch_size, col:col+patch_size]

            # Skip patches with too many NaN or zero values (land pixels)
            valid_pixels = np.sum(np.isfinite(patch) & (patch != 0))
            total_pixels = patch_size * patch_size
            if valid_pixels / total_pixels < 0.75:
                continue  # skip if more than 25% land — matches paper

            # Z-score normalise
            mean = np.nanmean(patch)
            std  = np.nanstd(patch)
            if std < 1e-6:
                continue  # skip uniform patches
            patch_norm = (patch - mean) / std

            patches.append(patch_norm)
            coords.append((row + patch_size//2, col + patch_size//2))

    if len(patches) == 0:
        return np.array([]), []

    patches_array = np.array(patches, dtype=np.float32)
    patches_array = patches_array[..., np.newaxis]  # add channel dim → (N,49,49,1)
    return patches_array, coords


def augment_patches(patches, labels):
    """
    Augment dataset by rotating and flipping each patch.
    Matches paper: rotations 90, 180, 270 + left-right + up-down flip.
    
    patches: (N, 49, 49, 1)
    labels:  (N,) wind directions in radians
    
    Returns augmented arrays ~6x larger.
    """
    aug_patches = [patches]
    aug_labels  = [labels]

    # Rotations: 90, 180, 270 degrees
    # When image rotates by angle α, wind direction shifts by α too
    for k, angle_shift in zip([1, 2, 3], [np.pi/2, np.pi, 3*np.pi/2]):
        rotated = np.rot90(patches, k=k, axes=(1, 2))
        new_labels = (labels + angle_shift) % np.pi  # keep in [0, π] for aliased direction
        aug_patches.append(rotated)
        aug_labels.append(new_labels)

    # Flip left-right: reverses wind direction sign
    flipped_lr = patches[:, :, ::-1, :]
    aug_patches.append(flipped_lr)
    aug_labels.append((-labels) % np.pi)

    # Flip up-down: shifts direction by π
    flipped_ud = patches[:, ::-1, :, :]
    aug_patches.append(flipped_ud)
    aug_labels.append((labels + np.pi) % np.pi)

    return np.concatenate(aug_patches), np.concatenate(aug_labels)


# Quick test on fake data
fake_image = np.random.randn(500, 500).astype(np.float32)
patches, coords = extract_patches(fake_image)
print(f"Patches extracted: {len(patches)}")
print(f"Patch shape:       {patches[0].shape}")

# Test augmentation
fake_labels = np.random.uniform(0, np.pi, len(patches)).astype(np.float32)
aug_p, aug_l = augment_patches(patches, fake_labels)
print(f"After augmentation: {len(aug_p)} patches")
print(f"Expected ~6x:       {len(patches) * 6}")

In [ ]:
import numpy as np

def resolve_ambiguity(aliased_directions, era5_directions):
    """
    Resolve 180° ambiguity in ResNet output.
    
    For each aliased direction θ, two candidates exist: θ and θ + 180°.
    We pick the one forming the smallest angle with ERA5 direction.
    
    aliased_directions : numpy array of angles in degrees (0–180°)
    era5_directions    : numpy array of ERA5 angles in degrees (0–360°)
    
    Returns: de-aliased directions in degrees (0–360°)
    """
    resolved = np.zeros_like(aliased_directions)

    for i, (aliased, era5) in enumerate(zip(aliased_directions, era5_directions)):
        # Two candidates
        candidate1 = aliased % 360
        candidate2 = (aliased + 180) % 360

        # Angular difference with ERA5
        diff1 = abs(((candidate1 - era5 + 180) % 360) - 180)
        diff2 = abs(((candidate2 - era5 + 180) % 360) - 180)

        # Pick the one closer to ERA5
        resolved[i] = candidate1 if diff1 <= diff2 else candidate2

    return resolved


# Test it
test_aliased = np.array([45.0, 90.0, 135.0, 30.0])
test_era5    = np.array([50.0, 260.0, 140.0, 220.0])
result = resolve_ambiguity(test_aliased, test_era5)

print("Aliased directions:", test_aliased)
print("ERA5 directions:   ", test_era5)
print("Resolved:          ", result)
print()
# Expected: 45→45 (close to 50), 90→270 (close to 260), 135→135 (close to 140), 30→210 (close to 220)